In [ ]:
import sys
print(sys.executable)

In [ ]:
import pandas, numpy, matplotlib, lightgbm, sklift
for m in [pandas, numpy, matplotlib, lightgbm, sklift]:
    print(m.__name__, m.__version__)


In [ ]:
# Hillstromデータの読み込み
import glob
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("bofulee/kevin-hillstrom-minethatdata-e-mailanalytics")
print(path)
csv_files = glob.glob(f"{path}/**/*.csv", recursive=True)
print(csv_files)

In [ ]:
df = pd.read_csv(csv_files[0])
pd.set_option("display.html.use_mathjax", False)

print(df.shape)   # (行数, 列数)
df.head()         # 先頭5行

In [ ]:
df.info()

In [ ]:
df["segment"].value_counts()

In [ ]:
df["zip_code"] = df["zip_code"].replace({"Surburban": "Suburban"})
df["zip_code"].value_counts()

In [ ]:
## 平均効果を確認
# 群ごとの来訪率・購入率・平均購入額
summary = df.groupby("segment")[["visit", "conversion", "spend"]].mean()
print(summary.round(4))

# 配信なしとの差（＝平均効果）
base = summary.loc["No E-Mail"]
print("\n配信なしとの差")
print((summary - base).round(4))

In [ ]:
# 群ごとの人数・購入者数・売上合計
check = df.groupby("segment").agg(
    人数=("visit", "size"),
    来訪者数=("visit", "sum"),
    購入者数=("conversion", "sum"),
    売上合計=("spend", "sum"),
)
check["購入者1人あたり金額"] = check["売上合計"] / check["購入者数"]
check.round(1)

In [ ]:
## 効果のばらつきを確認
def uplift_by(col):
    t = df.pivot_table(index=col, columns="segment", values="visit", aggfunc="mean")
    out = pd.DataFrame({
        "人数": df.groupby(col).size(),
        "男性向けメールの効果": t["Mens E-Mail"] - t["No E-Mail"],
        "女性向けメールの効果": t["Womens E-Mail"] - t["No E-Mail"],
    })
    return out.round(4)

for col in ["mens", "womens", "newbie", "zip_code", "channel"]:
    print(f"=== {col} ===")
    print(uplift_by(col), "\n")

In [ ]:
# メールの効果はVisit数
# 組み合わせごとの人数
print(pd.crosstab(df["mens"], df["womens"], margins=True), "\n")

# 組み合わせごとの効果
df["購入履歴"] = df["mens"].astype(str) + "_" + df["womens"].astype(str)
print(uplift_by("購入履歴"))

In [ ]:
# conversionでメール効果を計算する
def uplift_by(col, target="visit"):
    t = df.pivot_table(index=col, columns="segment", values=target, aggfunc="mean")
    out = pd.DataFrame({
        "人数": df.groupby(col).size(),
        "男性向けメールの効果": t["Mens E-Mail"] - t["No E-Mail"],
        "女性向けメールの効果": t["Womens E-Mail"] - t["No E-Mail"],
    })
    return out.round(4)

print(uplift_by("購入履歴", target="conversion"))
print(uplift_by("購入履歴", target="spend"))

In [ ]:
# 購入金額と最終購入時期で効果を見る
for col in ["history_segment", "recency"]:
    print(f"=== {col} ===")
    print(uplift_by(col), "\n")

In [ ]:
# ランダム化の確認
import numpy as np

# 女性向けメール vs 配信なし に絞る
df2 = df[df["segment"].isin(["Womens E-Mail", "No E-Mail"])].copy()
df2["t"] = (df2["segment"] == "Womens E-Mail").astype(int)

# 文字列の列を0/1に分解（比較のため）
X = pd.get_dummies(
    df2[["recency", "history", "mens", "womens", "newbie", "zip_code", "channel"]],
    dtype=int,
)

# 標準化平均差（SMD）
treat, ctrl = X[df2["t"] == 1], X[df2["t"] == 0]
smd = (treat.mean() - ctrl.mean()) / np.sqrt((treat.var() + ctrl.var()) / 2)

result = pd.DataFrame({
    "女性向けメール群の平均": treat.mean(),
    "配信なし群の平均": ctrl.mean(),
    "SMD": smd,
}).round(3)
result

In [ ]:
print(treat.var().round(4))
print(ctrl.var().round(4))

In [ ]:
print(treat.shape)   # (女性向けメール群の人数, 属性の数)
print(ctrl.shape)    # (配信なし群の人数, 属性の数)
treat.head()

In [ ]:
## 信頼区間を表示するコード

def uplift_ci(col, email="Womens E-Mail", target="visit"):
    rows = []
    for v, g in df.groupby(col):
        a = g.loc[g["segment"] == email, target]
        b = g.loc[g["segment"] == "No E-Mail", target]
        diff = a.mean() - b.mean()
        se = np.sqrt(a.var() / len(a) + b.var() / len(b))
        rows.append({col: v, "効果": diff,
                     "下限": diff - 1.96 * se, "上限": diff + 1.96 * se})
    return pd.DataFrame(rows).round(4)

for col in ["購入履歴", "history_segment", "recency"]:
    print(uplift_ci(col), "\n")

In [ ]:
import matplotlib.pyplot as plt
import japanize_matplotlib

# 女性向けメールの全体平均効果
w = df.loc[df["segment"] == "Womens E-Mail", "visit"].mean()
n = df.loc[df["segment"] == "No E-Mail", "visit"].mean()
ate = w - n

def plot_ci(col, ax):
    r = uplift_ci(col)
    y = list(range(len(r)))
    ax.errorbar(
        r["効果"], y,
        xerr=[r["効果"] - r["下限"], r["上限"] - r["効果"]],
        fmt="o", capsize=4,
    )
    # $ が数式扱いされないようにエスケープ
    labels = r[col].astype(str).str.replace("$", r"\$", regex=False)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.axvline(ate, color="red", linestyle="--", label=f"全体平均 {ate:.3f}")
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.invert_yaxis()
    ax.set_title(col)
    ax.set_xlabel("女性向けメールの効果（来訪率の差）")
    # ax.legend(loc="lower right")

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for col, ax in zip(["購入履歴", "history_segment", "recency"], axes):
    plot_ci(col, ax)

# 凡例をグラフ全体の上に1つだけ表示
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=1, fontsize=11)

plt.tight_layout(rect=[0, 0, 1, 0.94])  # 上に凡例用の余白を確保
plt.savefig("uplift_ci_womens.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
## 他の指標でも検証

def uplift_ci(col, data=df, email="Womens E-Mail", target="visit"):
    rows = []
    for v, g in data.groupby(col):
        a = g.loc[g["segment"] == email, target]
        b = g.loc[g["segment"] == "No E-Mail", target]
        diff = a.mean() - b.mean()
        se = np.sqrt(a.var() / len(a) + b.var() / len(b))
        rows.append({col: v, "人数": len(g), "効果": diff,
                     "下限": diff - 1.96 * se, "上限": diff + 1.96 * se})
    return pd.DataFrame(rows).round(4)

def plot_ci(col, ax, data=df, ref=None, title=None):
    r = uplift_ci(col, data=data)
    y = list(range(len(r)))
    ax.errorbar(r["効果"], y,
                xerr=[r["効果"] - r["下限"], r["上限"] - r["効果"]],
                fmt="o", capsize=4)
    labels = r[col].astype(str).str.replace("$", r"\$", regex=False)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    if ref is not None:
        ax.axvline(ref, color="red", linestyle="--", label=f"基準 {ref:.3f}")
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.invert_yaxis()
    ax.set_title(title or col)
    ax.set_xlabel("女性向けメールの効果（来訪率の差）")

# womens=1 の人だけに絞る
w1 = df[df["womens"] == 1]
ref_w1 = (w1.loc[w1["segment"] == "Womens E-Mail", "visit"].mean()
          - w1.loc[w1["segment"] == "No E-Mail", "visit"].mean())

cols = ["zip_code", "channel", "newbie", "history_segment"]
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
for col, ax in zip(cols, axes):
    plot_ci(col, ax, data=w1, ref=ref_w1, title=f"{col}（womens=1 のみ）")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center")
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig("uplift_ci_womens_2.png", dpi=150, bbox_inches="tight")
plt.show()



In [ ]:
# モデルを作る
## 「この人にメールを送ったら、来訪する確率が何ポイント上がるか」を、一人ひとりについて予測する

from sklearn.model_selection import train_test_split

# 1. 女性向けメール vs 配信なし に絞る
data = df[df["segment"].isin(["Womens E-Mail", "No E-Mail"])].copy()
data["t"] = (data["segment"] == "Womens E-Mail").astype(int)

# 2. 説明変数を作る（history_segment は history と重複するので使わない）
features = ["recency", "history", "mens", "womens", "newbie", "zip_code", "channel"]
X = pd.get_dummies(data[features], columns=["zip_code", "channel"], dtype=int)
y = data["visit"]
t = data["t"]

# 3. 学習用70%・評価用30%に分ける
#    「処置 × 来訪」の4パターンの比率がそろうように層化する
strata = t.astype(str) + "_" + y.astype(str)
X_train, X_test, y_train, y_test, t_train, t_test = train_test_split(
    X, y, t, test_size=0.3, random_state=42, stratify=strata
)

print(X_train.shape, X_test.shape)
print("学習用の処置群の割合:", t_train.mean().round(3))
print("評価用の処置群の割合:", t_test.mean().round(3))
X_train.head()

## このモデルでやろうとしていること
一言で言うと、「この人にメールを送ったら、来訪する確率が何ポイント上がるか」を、一人ひとりについて予測することです。

これまでの手作業との関係
これまでは「womens=1 の人」「Rural の人」のように、グループごとに効果を計算してきました。モデルは同じことを、すべての属性の組み合わせについて同時にやります。「女性向け商品の購入歴あり・地方在住・購入金額300ドル・最終購入3か月前・Web で購入」のような細かい組み合わせごとに効果を推定するイメージです。手作業だと組み合わせが多すぎて人数が足りなくなりますが、モデルは似た人たちの情報を借りて推定します。

# 仕組み（T-learner で説明すると）
モデルA：メールが届いた人のデータで「どんな人が来訪するか」を学習
モデルB：メールが届かなかった人のデータで「どんな人が来訪するか」を学習

# ある人について
  モデルA の予測：メールが届いたら来訪する確率 → 例えば 15%
  モデルB の予測：メールが届かなかったら来訪する確率 → 例えば 9%
  差 = 6ポイント ← これがその人へのメールの効果（アップリフト）

同じ人について「送った場合」と「送らなかった場合」を両方観測することはできないので、2つのモデルで両方の世界を予測して、差を取るわけです。

# 最後にやること
評価用データの一人ひとりに、効果の予測値（スコア）を付ける
スコアの高い順に並べる
上位の人たちで、実際に「メールあり群とメールなし群の来訪率の差」が大きくなっているかを確かめる

上位ほど実際の差が大きければ、モデルは「メールが効く人」を正しく見分けられている、ということになります。

# ビジネス上のアウトプット
最終的には「スコアの上位◯%に送る」という配信リストになります。そして Step 5 で、「上位の人はどんな人か」を言葉にすることで、「誰に送るべきか」の答えになります。

## ここから Step 3（モデル構築）です。今回の作業は次の3つです。
・3つの手法でモデルを作る
・評価用データの一人ひとりに、効果の予測値（スコア）を付ける
・スコアがおかしな値になっていないかを確認する

In [ ]:
# セル1：共通設定
import lightgbm as lgb

params = dict(
    n_estimators=200,       # 木の数
    learning_rate=0.05,     # 1本ごとの学習の歩幅
    num_leaves=15,          # 木の複雑さ（小さめに抑える）
    min_child_samples=100,  # 1つの葉に最低100人（細かく分けすぎない）
    random_state=42,
    verbose=-1,             # 学習中のメッセージを非表示
)

In [ ]:
# セル2：S-learner
# 「メールが届いたか（t）」を特徴量の1つとして、1つのモデルで学習
s_model = lgb.LGBMClassifier(**params)
s_model.fit(X_train.assign(t=t_train.values), y_train)

# 評価用データの全員について「届いた場合」と「届かなかった場合」を予測し、差を取る
p1 = s_model.predict_proba(X_test.assign(t=1))[:, 1]
p0 = s_model.predict_proba(X_test.assign(t=0))[:, 1]
uplift_s = p1 - p0

In [ ]:
# セル3：T-learner
# メールが届いた人だけでモデルA、届かなかった人だけでモデルBを学習
m1 = lgb.LGBMClassifier(**params).fit(X_train[t_train == 1], y_train[t_train == 1])
m0 = lgb.LGBMClassifier(**params).fit(X_train[t_train == 0], y_train[t_train == 0])

uplift_t = m1.predict_proba(X_test)[:, 1] - m0.predict_proba(X_test)[:, 1]

In [ ]:
# セル4：Class Transformation
# 目的変数を変換する
#   z = 1：「メールが届いて来訪した」または「届かずに来訪しなかった」
#   z = 0：それ以外
z_train = (y_train == t_train).astype(int)

ct_model = lgb.LGBMClassifier(**params).fit(X_train, z_train)
uplift_ct = 2 * ct_model.predict_proba(X_test)[:, 1] - 1

In [ ]:
# セル5：スコアの確認
scores = pd.DataFrame({
    "S-learner": uplift_s,
    "T-learner": uplift_t,
    "Class Transformation": uplift_ct,
}, index=X_test.index)

# 評価用データでの実際の平均効果
actual = y_test[t_test == 1].mean() - y_test[t_test == 0].mean()
print("評価用データの実際の平均効果:", round(actual, 4), "\n")

print(scores.describe().round(4), "\n")
print("手法間の相関")
print(scores.corr().round(3))

In [ ]:
## 予測スコアを womens 別・Rural 別に集計
# 評価用データの実際の効果（グループ別）
def actual_uplift(mask):
    yt = y_test[mask & (t_test == 1)].mean()
    yc = y_test[mask & (t_test == 0)].mean()
    return yt - yc

for col in ["womens", "zip_code_Rural"]:
    print(f"=== {col} ===")
    table = scores.groupby(X_test[col]).mean()
    table["実際の効果（評価用）"] = [
        actual_uplift(X_test[col] == v) for v in table.index
    ]
    table["人数"] = X_test[col].value_counts().sort_index()
    print(table.round(4), "\n")

In [ ]:
# 準備
rng = np.random.default_rng(42)

eval_scores = scores.copy()
# 単純ルール：womens=1 なら1、0なら0
# （同じ値の人の順番がランダムになるよう、ごく小さな乱数を足している）
eval_scores["単純ルール（womens）"] = (
    X_test["womens"].values + rng.uniform(0, 1e-6, len(X_test))
)

yv = y_test.values
tv = t_test.values

In [ ]:
# 十分位別の実際の効果
def decile_table(score):
    # スコアの高い順に10グループに分ける（1が上位10%）
    order = np.argsort(-score)
    bins = np.empty(len(score), dtype=int)
    bins[order] = np.arange(len(score)) * 10 // len(score) + 1

    d = pd.DataFrame({"bin": bins, "y": yv, "t": tv})
    rate = d.pivot_table(index="bin", columns="t", values="y", aggfunc="mean")
    out = pd.DataFrame({
        "人数": d.groupby("bin").size(),
        "メールあり来訪率": rate[1],
        "メールなし来訪率": rate[0],
    })
    out["実際の効果"] = out["メールあり来訪率"] - out["メールなし来訪率"]
    return out.round(4)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for name, ax in zip(eval_scores.columns, axes.flatten()):
    tbl = decile_table(eval_scores[name].values)
    ax.bar(tbl.index, tbl["実際の効果"])
    ax.axhline(actual, color="red", linestyle="--", label=f"全体平均 {actual:.3f}")
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel("スコアの順位グループ（1＝上位10%）")
    ax.set_ylabel("実際の効果（来訪率の差）")
    ax.set_xticks(range(1, 11))
    ax.legend()
plt.tight_layout()
fig.savefig("step4_decile.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Qini曲線

def qini_curve(score):
    order = np.argsort(-score)
    y_s, t_s = yv[order], tv[order]
    nt = np.cumsum(t_s)              # 上位から数えたメールあり群の人数
    nc = np.cumsum(1 - t_s)          # 上位から数えたメールなし群の人数
    yt = np.cumsum(y_s * t_s)        # メールあり群の来訪者数
    yc = np.cumsum(y_s * (1 - t_s))  # メールなし群の来訪者数
    # メールなし群の来訪者数を、メールあり群の人数に合わせて換算して差を取る
    q = yt - np.divide(yc * nt, nc, out=np.zeros(len(nc), dtype=float), where=nc > 0)
    x = np.arange(1, len(score) + 1) / len(score)
    return np.r_[0, x], np.r_[0, q]

fig, ax = plt.subplots(figsize=(9, 7))
results = {}
for name in eval_scores.columns:
    x, q = qini_curve(eval_scores[name].values)
    auc = np.trapezoid(q, x) - q[-1] / 2   # ランダム配信の直線との間の面積
    results[name] = auc
    ax.plot(x, q, label=f"{name}（Qini AUC {auc:.1f}）")

ax.plot([0, 1], [0, q[-1]], color="gray", linestyle="--", label="ランダム配信")
ax.set_xlabel("配信する人の割合（スコア上位から）")
ax.set_ylabel("増分来訪者数（累積）")
ax.set_title("Qini曲線（女性向けメール）")
ax.legend()
fig.savefig("step4_qini.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 上位30%に送った場合の効果（uplift@30%）
def uplift_at_k(score, k=0.3):
    top = np.argsort(-score)[: int(len(score) * k)]
    y_k, t_k = yv[top], tv[top]
    return y_k[t_k == 1].mean() - y_k[t_k == 0].mean()

summary = pd.DataFrame({
    "Qini AUC": results,
    "uplift@30%": {n: uplift_at_k(eval_scores[n].values) for n in eval_scores.columns},
}).round(4)
summary

In [ ]:
# 上位30%・中間・下位30%の比較
s = scores["S-learner"]
q70, q30 = s.quantile(0.7), s.quantile(0.3)
grp = np.where(s >= q70, "上位30%", np.where(s <= q30, "下位30%", "中間"))

profile = X_test.groupby(grp).mean().T[["上位30%", "中間", "下位30%"]]
profile.round(3)

In [ ]:
# 決定木でルール化
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text

# S-learner のスコアを、属性からどう説明できるかを浅い木で表す
tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=500, random_state=42)
tree.fit(X_test, s)

print("木で説明できたスコアの割合（R²）:", round(tree.score(X_test, s), 3), "\n")
print(export_text(tree, feature_names=list(X_test.columns), decimals=3))

In [ ]:
# 決定木を図にする
fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(tree, feature_names=list(X_test.columns), filled=True,
          rounded=True, precision=3, fontsize=10, ax=ax)
fig.savefig("step5_tree.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
for depth in [1, 2, 3]:
    t_ = DecisionTreeRegressor(max_depth=depth, min_samples_leaf=500, random_state=42)
    t_.fit(X_test, s)
    print(f"深さ{depth}：R² = {t_.score(X_test, s):.3f}")

In [ ]:
# 4象限の整理（女性向けメール）
base = df[df["segment"] == "No E-Mail"].groupby("購入履歴")["visit"].mean()
quad = pd.DataFrame({
    "人数": df.groupby("購入履歴").size(),
    "メールなしの来訪率": base,
    "女性向けメールの効果": uplift_by("購入履歴")["女性向けメールの効果"],
    "男性向けメールの効果": uplift_by("購入履歴")["男性向けメールの効果"],
})
quad.round(4)

In [ ]:
# 配信シナリオの比較
shares = df["購入履歴"].value_counts(normalize=True)

def seg_uplift(target):
    t = df.pivot_table(index="購入履歴", columns="segment", values=target, aggfunc="mean")
    return pd.DataFrame({e: t[e] - t["No E-Mail"] for e in ["Mens E-Mail", "Womens E-Mail"]})

U = {m: seg_uplift(m) for m in ["visit", "conversion", "spend"]}

W, M = "Womens E-Mail", "Mens E-Mail"
policies = {
    "①全員に女性向け":            {"0_1": W, "1_0": W,    "1_1": W},
    "②女性向け購入歴ありにだけ女性向け": {"0_1": W, "1_0": None, "1_1": W},
    "③全員に男性向け":            {"0_1": M, "1_0": M,    "1_1": M},
    "④購入履歴で出し分け":        {"0_1": W, "1_0": M,    "1_1": M},
}

rows = []
for name, pol in policies.items():
    r = {"施策": name}
    r["配信率"] = sum(shares[s] for s, e in pol.items() if e)
    for m, label in [("visit", "増分来訪"), ("conversion", "増分購入"), ("spend", "増分売上（ドル）")]:
        r[label] = sum(shares[s] * U[m].loc[s, e] for s, e in pol.items() if e) * 10000
    r["1通あたり増分売上（ドル）"] = r["増分売上（ドル）"] / (r["配信率"] * 10000)
    rows.append(r)

scenario = pd.DataFrame(rows).set_index("施策")
scenario.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for email, col, mk in [("女性向けメール", "女性向けメールの効果", "o"),
                       ("男性向けメール", "男性向けメールの効果", "s")]:
    ax.scatter(quad["メールなしの来訪率"], quad[col], s=quad["人数"] / 50,
               marker=mk, alpha=0.6, label=email)
    for seg in quad.index:
        ax.annotate(seg, (quad.loc[seg, "メールなしの来訪率"], quad.loc[seg, col]),
                    xytext=(8, 0), textcoords="offset points")
ax.axhline(quad["女性向けメールの効果"].mean(), color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("メールなしの来訪率（送らなくても来るか）")
ax.set_ylabel("メールの効果（送ると増えるか）")
ax.set_title("顧客タイプ × メール内容の4象限")
ax.legend()
fig.savefig("step6_quadrant.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sc = scenario.copy()
sc["売上（①との比）"] = sc["増分売上（ドル）"] / sc.loc["①全員に女性向け", "増分売上（ドル）"]

(sc.style
   .format({"配信率": "{:.0%}", "増分来訪": "{:,.0f}人", "増分購入": "{:,.0f}人",
            "増分売上（ドル）": "${:,.0f}", "1通あたり増分売上（ドル）": "${:.2f}",
            "売上（①との比）": "{:.2f}倍"})
   .bar(subset=["増分売上（ドル）"], color="#9ecae1")
   .bar(subset=["1通あたり増分売上（ドル）"], color="#fdae6b")
   .set_caption("配信シナリオ比較（顧客1万人あたり）"))

In [ ]:
# 結果の視覚化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names = sc.index

axes[0].barh(names, sc["増分売上（ドル）"], color="#4C78A8")
axes[0].set_title("増分売上（顧客1万人あたり）")
axes[0].bar_label(axes[0].containers[0], fmt="$%.0f", padding=3)

axes[1].barh(names, sc["1通あたり増分売上（ドル）"], color="#F58518")
axes[1].set_title("1通あたり増分売上（配信の効率）")
axes[1].bar_label(axes[1].containers[0], fmt="$%.2f", padding=3)

for ax in axes:
    ax.invert_yaxis()
    ax.set_xlim(0, ax.get_xlim()[1] * 1.15)  # ラベル用の余白
plt.tight_layout()
fig.savefig("step6_scenario.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 結果の視覚化
labels_map = {"0_1": "女性向けのみ購入", "1_0": "男性向けのみ購入", "1_1": "両方購入"}
q = quad.rename(index=labels_map)

x = np.arange(len(q))
w = 0.38
fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - w/2, q["女性向けメールの効果"] * 100, w, label="女性向けメール", color="#F58518")
b2 = ax.bar(x + w/2, q["男性向けメールの効果"] * 100, w, label="男性向けメール", color="#4C78A8")
ax.bar_label(b1, fmt="%.1fpt", padding=3)
ax.bar_label(b2, fmt="%.1fpt", padding=3)

ax.set_xticks(x)
ax.set_xticklabels([
    f"{name}\n{int(n):,}人／メールなし来訪率 {r:.0%}"
    for name, n, r in zip(q.index, q["人数"], q["メールなしの来訪率"])
])
ax.set_ylabel("来訪率の増分（ポイント）")
ax.set_title("顧客タイプ × メール内容ごとの効果")
ax.legend()
ax.set_ylim(0, q[["女性向けメールの効果", "男性向けメールの効果"]].max().max() * 100 * 1.15)
fig.savefig("step6_who_x_what.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 4象限概念図
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axhline(0.5, color="gray"); ax.axvline(0.5, color="gray")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xticks([]); ax.set_yticks([])

cells = [
    (0.25, 0.75, "説得可能層\n- Persuadables -", "広告を見ると行動する\n→ 届けるべき人", "#4C78A8"),
    (0.75, 0.75, "鉄板層\n- Sure Things -", "広告がなくても行動する\n→ 広告費の無駄", "#9ecae1"),
    (0.75, 0.25, "逆効果層\n- Sleeping Dogs -", "広告を見ると離れる\n→ 届けてはいけない", "#E45756"),
    (0.25, 0.25, "無関心層\n- Lost Causes -", "広告を見ても行動しない\n→ 効果がない", "#bab0ac"),
]
for x, y, title, desc, c in cells:
    ax.text(x, y + 0.06, title, ha="center", fontsize=16, fontweight="bold", color=c)
    ax.text(x, y - 0.06, desc, ha="center", va="center", fontsize=11)

ax.set_xlabel("広告がない場合 → 行動しない（左）／行動する（右）", fontsize=11)
ax.set_ylabel("広告がある場合 → 行動しない（下）／行動する（上）", fontsize=11)
ax.set_title("広告への反応による顧客の4分類", fontsize=14)
fig.savefig("fig0_quadrant_concept.png", dpi=150, bbox_inches="tight")
plt.show()